# K-Means Clustering - Aprendizaje No Supervisado

Bienvenido al mundo del **aprendizaje no supervisado**. K-Means es el algoritmo de clustering más utilizado, capaz de encontrar grupos naturales en datos sin etiquetas.

Al finalizar este notebook, serás capaz de:

* Comprender la diferencia entre aprendizaje supervisado y no supervisado
* Implementar K-Means desde cero usando solo NumPy
* Entender el algoritmo iterativo: asignación y actualización de centroides
* Aplicar el método del codo (Elbow Method) para seleccionar K óptimo
* Calcular y optimizar la inercia (within-cluster sum of squares)
* Reconocer limitaciones: sensibilidad a inicialización y forma de clusters
* Comparar K-Means con otros algoritmos de clustering
* Aplicar K-Means a problemas reales de segmentación

**¿Por qué K-Means?**

1. **Simplicidad**: Algoritmo intuitivo y fácil de implementar
2. **Escalabilidad**: Funciona bien con grandes datasets
3. **Velocidad**: Convergencia rápida en la mayoría de casos
4. **Versatilidad**: Aplicable en muchos dominios (segmentación, compresión, etc.)
5. **Fundamento**: Base para entender algoritmos más complejos (GMM, clustering jerárquico)

**Diferencias con algoritmos supervisados:**
- **Supervisado** (Regresión, Clasificación): Tenemos etiquetas $y$ para entrenar
- **No Supervisado** (Clustering): NO tenemos etiquetas, buscamos estructura oculta en los datos

## Nota Importante sobre los Ejercicios

Antes de comenzar con los ejercicios, ten en cuenta lo siguiente:

1. NO agregues declaraciones `print` adicionales en las funciones graduadas
2. NO agregues celdas de código adicionales entre los ejercicios
3. NO cambies los parámetros de las funciones
4. Implementa usando NumPy (operaciones vectorizadas cuando sea posible)
5. NO cambies el código de las pruebas automáticas

Si experimentas errores al ejecutar las pruebas, primero verifica estos puntos antes de buscar ayuda.

<a name='1'></a>
## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Teoría de K-Means Clustering](#2)
- [3 - Implementación desde Cero](#3)
  - [Ejercicio 1](#ex01)
- [4 - Clustering en Acción](#4)
- [5 - Selección del Número de Clusters K](#5)
  - [Ejercicio 2](#ex02)
- [6 - Referencias](#6)

<a name='1'></a>
## 1 - Paquetes

Ejecuta la siguiente celda para importar los paquetes que usarás en este notebook:

* **NumPy**: Operaciones numéricas y cálculo de distancias
* **Matplotlib**: Visualización de clusters y centroides
* **Scikit-learn**: Generación de datos sintéticos (make_blobs)
* **Testing utilities**: Verificación automática de ejercicios

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from sklearn.datasets import make_blobs

# Configurar matplotlib inline
%matplotlib inline

# Agregar el directorio raíz al path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Importar utilidades de testing
from utils.testing_utils import print_success, print_error, print_info
from tests.no_supervisados.test_01_kmeans import (
    test_ejercicio_1_assign_clusters,
    test_ejercicio_2_update_centroids
)

print("✅ Paquetes importados correctamente")
print(f"📦 NumPy version: {np.__version__}")

<a name='2'></a>
## 2 - Teoría de K-Means Clustering

K-Means es un algoritmo de **clustering no supervisado** que particiona un conjunto de datos en $K$ grupos (clusters) basándose en la similitud.

### 2.1 - Problema de Clustering

**Dado:** Un conjunto de datos $\mathbf{X} = \{\mathbf{x}^{(1)}, \mathbf{x}^{(2)}, ..., \mathbf{x}^{(m)}\}$ SIN etiquetas

**Objetivo:** Encontrar $K$ grupos (clusters) tales que los puntos dentro de cada cluster sean similares entre sí y diferentes de otros clusters

### 2.2 - Algoritmo de K-Means

El algoritmo es iterativo y consiste en dos pasos que se repiten:

**Inicialización:** Seleccionar $K$ centroides iniciales $\{\mu_1, \mu_2, ..., \mu_K\}$ aleatoriamente

**Paso 1 - Asignación:** Asignar cada punto $\mathbf{x}^{(i)}$ al cluster del centroide más cercano:

$$c^{(i)} = \underset{k}{\text{argmin}} \, \|\mathbf{x}^{(i)} - \mu_k\|^2$$

donde $c^{(i)} \in \{1, 2, ..., K\}$ es el cluster asignado al punto $i$.

**Paso 2 - Actualización:** Recalcular cada centroide como la media de los puntos asignados:

$$\mu_k = \frac{1}{|C_k|} \sum_{\mathbf{x}^{(i)} \in C_k} \mathbf{x}^{(i)}$$

donde $C_k = \{\mathbf{x}^{(i)} : c^{(i)} = k\}$ es el conjunto de puntos asignados al cluster $k$.

**Criterio de parada:** Repetir Paso 1 y 2 hasta que:
- Los centroides no cambien significativamente, O
- Se alcance un número máximo de iteraciones

### 2.3 - Función Objetivo (Inercia)

K-Means minimiza la **inercia** o **within-cluster sum of squares (WCSS)**:

$$J(\mathbf{c}, \mu) = \sum_{i=1}^{m} \|\mathbf{x}^{(i)} - \mu_{c^{(i)}}\|^2 = \sum_{k=1}^{K} \sum_{\mathbf{x}^{(i)} \in C_k} \|\mathbf{x}^{(i)} - \mu_k\|^2$$

**Interpretación:** Suma de las distancias al cuadrado entre cada punto y el centroide de su cluster.

**Propiedad importante:** El algoritmo K-Means está garantizado a converger (la inercia nunca aumenta), pero puede converger a un **mínimo local**, no necesariamente al mínimo global.

### 2.4 - Propiedades y Garantías

**Convergencia:** K-Means siempre converge, pero:
- Puede converger a un mínimo local
- El resultado depende de la inicialización
- Solución: ejecutar múltiples veces con diferentes inicializaciones

**Complejidad:** $O(K \cdot m \cdot d \cdot t)$ donde:
- $K$ = número de clusters
- $m$ = número de puntos
- $d$ = dimensionalidad
- $t$ = número de iteraciones

**Limitaciones:**
- Asume clusters esféricos (forma circular/esférica)
- Sensible a outliers
- Necesita especificar $K$ a priori
- Sensible a la escala de features (normalizar!)

<a name='3'></a>
## 3 - Implementación desde Cero

Vamos a implementar K-Means paso a paso.

### 3.1 - Estructura de la Clase

Nuestra clase `KMeans` tendrá:
* `fit(X)`: Ejecuta el algoritmo K-Means
* `_assign_clusters(X)`: Paso 1 - Asigna puntos a clusters
* `_update_centroids(X)`: Paso 2 - Actualiza centroides
* `_calculate_inertia(X)`: Calcula la función objetivo
* `predict(X)`: Predice clusters para nuevos datos
* `inertia_`: Propiedad que retorna la inercia final

### 3.2 - Código de Implementación

In [ ]:
class KMeans:
    """
    K-Means Clustering implementado desde cero.
    
    Parámetros:
    -----------
    n_clusters : int
        Número de clusters K
    max_iters : int
        Número máximo de iteraciones
    random_state : int, optional
        Semilla para reproducibilidad
    """
    
    def __init__(self, n_clusters=3, max_iters=100, random_state=None):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_history = []
    
    def fit(self, X):
        """
        Ejecuta el algoritmo K-Means.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Datos a agrupar
        
        Returns:
        --------
        self
        """
        if self.random_state is not None:
            np.random.seed(self.random_state)
        
        X = np.array(X)
        n_samples, n_features = X.shape
        
        # Inicialización: seleccionar K puntos aleatorios como centroides
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices].copy()
        
        for iteration in range(self.max_iters):
            # Paso 1: Asignar cada punto al centroide más cercano
            self.labels = self._assign_clusters(X)
            
            # Guardar centroides anteriores para verificar convergencia
            old_centroids = self.centroids.copy()
            
            # Paso 2: Actualizar centroides
            self.centroids = self._update_centroids(X)
            
            # Calcular inercia (función objetivo)
            inertia = self._calculate_inertia(X)
            self.inertia_history.append(inertia)
            
            # Verificar convergencia: si centroides no cambian, parar
            if np.allclose(old_centroids, self.centroids, atol=1e-6):
                print(f"✅ Convergencia alcanzada en iteración {iteration+1}")
                break
        else:
            print(f"⚠️  Alcanzado número máximo de iteraciones ({self.max_iters})")
        
        return self
    
    def _assign_clusters(self, X):
        """
        Asigna cada punto al cluster más cercano (Paso 1).
        
        Parámetros:
        -----------
        X : ndarray, shape (n_samples, n_features)
        
        Returns:
        --------
        ndarray, shape (n_samples,)
            Índice del cluster asignado para cada punto
        """
        # Calcular distancias de cada punto a cada centroide
        distances = np.zeros((X.shape[0], self.n_clusters))
        
        for k, centroid in enumerate(self.centroids):
            distances[:, k] = np.linalg.norm(X - centroid, axis=1)
        
        # Asignar cada punto al centroide más cercano
        return np.argmin(distances, axis=1)
    
    def _update_centroids(self, X):
        """
        Actualiza centroides como media de puntos asignados (Paso 2).
        
        Parámetros:
        -----------
        X : ndarray, shape (n_samples, n_features)
        
        Returns:
        --------
        ndarray, shape (n_clusters, n_features)
            Nuevos centroides
        """
        new_centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for k in range(self.n_clusters):
            # Obtener puntos asignados al cluster k
            cluster_points = X[self.labels == k]
            
            if len(cluster_points) > 0:
                # Calcular media de los puntos
                new_centroids[k] = cluster_points.mean(axis=0)
            else:
                # Si un cluster está vacío, reinicializar con un punto aleatorio
                new_centroids[k] = X[np.random.choice(X.shape[0])]
        
        return new_centroids
    
    def _calculate_inertia(self, X):
        """
        Calcula la inercia (within-cluster sum of squares).
        
        Parámetros:
        -----------
        X : ndarray, shape (n_samples, n_features)
        
        Returns:
        --------
        float
            Inercia total
        """
        inertia = 0
        for k in range(self.n_clusters):
            cluster_points = X[self.labels == k]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - self.centroids[k]) ** 2)
        return inertia
    
    def predict(self, X):
        """
        Predice clusters para nuevos datos.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
        
        Returns:
        --------
        ndarray, shape (n_samples,)
            Cluster asignado para cada punto
        """
        return self._assign_clusters(np.array(X))
    
    @property
    def inertia_(self):
        """Retorna la inercia final"""
        return self.inertia_history[-1] if self.inertia_history else None

print("✅ Clase KMeans definida correctamente")

<a name='4'></a>
## 4 - Clustering en Acción

Vamos a aplicar K-Means a datos sintéticos para visualizar cómo funciona.

### 4.1 - Generar Datos Sintéticos

Usamos `make_blobs` para crear datos con 3 clusters bien definidos:

In [ ]:
# Generar datos con 3 clusters bien separados
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2, 
                       cluster_std=0.6, random_state=42)

# Visualizar datos originales (sin etiquetar)
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], s=50, alpha=0.6, edgecolors='k', c='gray')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos sin etiquetar - ¿Cuántos clusters hay?')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total de puntos: {len(X)}")
print(f"Dimensionalidad: {X.shape[1]}")
print("Nota: En problemas reales NO conocemos las etiquetas verdaderas")

### 4.2 - Entrenar K-Means

Ejecutamos K-Means con K=3 clusters:

In [ ]:
# Entrenar K-Means
kmeans = KMeans(n_clusters=3, max_iters=100, random_state=42)
kmeans.fit(X)

print(f"\n📊 Resultados:")
print(f"   Inercia final: {kmeans.inertia_:.2f}")
print(f"   Número de iteraciones: {len(kmeans.inertia_history)}")
print(f"\n🎯 Centroides encontrados:")
for i, centroid in enumerate(kmeans.centroids):
    print(f"   Cluster {i}: {centroid}")
    
print(f"\n📈 Distribución de puntos por cluster:")
unique, counts = np.unique(kmeans.labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"   Cluster {cluster_id}: {count} puntos ({count/len(X)*100:.1f}%)")

### 4.3 - Visualizar Clusters Encontrados

Graficamos los clusters identificados y sus centroides:

In [ ]:
# Visualizar clusters encontrados
def plot_kmeans_clusters(X, labels, centroids, title="K-Means Clustering"):
    """Grafica los clusters y centroides"""
    plt.figure(figsize=(10, 6))
    
    # Graficar puntos coloreados por cluster
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    for k in range(len(centroids)):
        cluster_points = X[labels == k]
        plt.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                   c=colors[k % len(colors)], label=f'Cluster {k}',
                   alpha=0.6, edgecolors='k', s=50)
    
    # Graficar centroides
    plt.scatter(centroids[:, 0], centroids[:, 1], 
               c='black', marker='X', s=300, linewidths=2,
               label='Centroides', edgecolors='yellow')
    
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_kmeans_clusters(X, kmeans.labels, kmeans.centroids,
                     title=f'K-Means Clustering (K={kmeans.n_clusters})')

### 4.4 - Convergencia del Algoritmo

Observamos cómo disminuye la inercia durante las iteraciones:

In [ ]:
# Visualizar convergencia
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(kmeans.inertia_history) + 1), kmeans.inertia_history, 
         linewidth=2, marker='o', markersize=6)
plt.xlabel('Iteración')
plt.ylabel('Inercia (WCSS)')
plt.title('Convergencia de K-Means')
plt.grid(True, alpha=0.3)
plt.show()

print("Observa cómo la inercia disminuye monótonamente hasta convergencia")

<a name='6'></a>
## 6 - Referencias

### Papers Fundamentales

1. **MacQueen, J.** (1967). "Some methods for classification and analysis of multivariate observations." *Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability*, 1(14), 281-297.
   - Introducción del algoritmo K-Means

2. **Lloyd, S.** (1982). "Least squares quantization in PCM." *IEEE Transactions on Information Theory*, 28(2), 129-137.
   - Algoritmo de Lloyd (equivalente a K-Means)

3. **Arthur, D., & Vassilvitskii, S.** (2007). "k-means++: The advantages of careful seeding." *Proceedings of the Eighteenth Annual ACM-SIAM Symposium on Discrete Algorithms*, 1027-1035.
   - K-Means++ con mejor inicialización

### Recursos Adicionales

4. **Bishop, C. M.** (2006). *Pattern Recognition and Machine Learning*. Springer. Chapter 9: Mixture Models and EM.
   - Tratamiento teórico de clustering

5. **Murphy, K. P.** (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press. Chapter 25: Clustering.
   - Perspectiva probabilística de K-Means

6. **James, G., Witten, D., Hastie, T., & Tibshirani, R.** (2013). *An Introduction to Statistical Learning*. Springer. Chapter 10: Unsupervised Learning.
   - Introducción accesible a clustering

### Métodos para Seleccionar K

7. **Thorndike, R. L.** (1953). "Who belongs in the family?" *Psychometrika*, 18(4), 267-276.
   - Método del codo original

8. **Rousseeuw, P. J.** (1987). "Silhouettes: a graphical aid to the interpretation and validation of cluster analysis." *Journal of Computational and Applied Mathematics*, 20, 53-65.
   - Silhouette coefficient para evaluar clustering

9. **Tibshirani, R., Walther, G., & Hastie, T.** (2001). "Estimating the number of clusters in a data set via the gap statistic." *Journal of the Royal Statistical Society: Series B*, 63(2), 411-423.
   - Gap statistic como alternativa al método del codo

### Variantes y Extensiones

10. **Sculley, D.** (2010). "Web-scale k-means clustering." *Proceedings of the 19th International Conference on World Wide Web*, 1177-1178.
    - Mini-Batch K-Means para big data

11. **Kaufman, L., & Rousseeuw, P. J.** (1987). "Clustering by means of medoids." *Statistical Data Analysis Based on the L1–Norm and Related Methods*, 405-416.
    - K-Medoids como alternativa robusta

12. **Bezdek, J. C.** (1981). *Pattern Recognition with Fuzzy Objective Function Algorithms*. Springer.
    - Fuzzy C-Means (soft clustering)

### Recursos Online

13. **Scikit-learn Documentation: K-Means**
    - https://scikit-learn.org/stable/modules/clustering.html#k-means
    - Implementaciones optimizadas y variantes

14. **StatQuest: K-means clustering**
    - https://www.youtube.com/watch?v=4b5d3muPQmA
    - Explicación visual intuitiva

15. **Google's Machine Learning Crash Course**
    - https://developers.google.com/machine-learning/clustering/overview
    - Tutorial interactivo sobre clustering

### Algoritmos Relacionados

16. **DBSCAN**: Density-based clustering (formas arbitrarias)
17. **Hierarchical Clustering**: Dendrogramas y aglomeración
18. **Gaussian Mixture Models (GMM)**: Clustering probabilístico
19. **Spectral Clustering**: Usa eigenvalues del grafo de similitud
20. **HDBSCAN**: Hierarchical DBSCAN más robusto

## 📘 Resumen y Aplicaciones en ML

<div style="background-color: #e7f3fe; padding: 20px; border-left: 6px solid #2196F3; margin: 20px 0;">

**Conceptos Clave Aprendidos:**

1. **Aprendizaje No Supervisado**: Encontrar estructura en datos sin etiquetas
2. **Algoritmo Iterativo**: Asignación → Actualización → Repetir
3. **Función Objetivo**: Minimizar inercia (WCSS)
4. **Método del Codo**: Técnica para seleccionar K óptimo
5. **Convergencia Garantizada**: Siempre converge, pero a mínimo local

**Ventajas de K-Means:**

✅ **Simple e intuitivo**: Fácil de entender e implementar  
✅ **Rápido**: Complejidad lineal en número de muestras  
✅ **Escalable**: Funciona bien con datasets grandes  
✅ **Interpretable**: Centroides son representativos de cada cluster  
✅ **Versátil**: Aplicable a muchos tipos de datos

**Desventajas de K-Means:**

❌ **Requiere especificar K**: Necesitas saber cuántos clusters  
❌ **Sensible a inicialización**: Puede converger a mínimos locales  
❌ **Asume clusters esféricos**: No funciona bien con formas irregulares  
❌ **Sensible a outliers**: Un outlier puede distorsionar centroides  
❌ **Sensible a escala**: Normalizar features es crítico

**Aplicaciones Reales:**

- **Segmentación de clientes**: Agrupar clientes por comportamiento
- **Compresión de imágenes**: Reducir colores manteniendo calidad
- **Detección de anomalías**: Puntos alejados de todos los centroides
- **Organización de documentos**: Agrupar textos similares
- **Agrupación geográfica**: Zonas de reparto, ubicaciones óptimas
- **Bioinformática**: Clasificación de genes, proteínas
- **Marketing**: Segmentación de mercado para campañas dirigidas

**Variantes y Mejoras:**

1. **K-Means++**: Inicialización inteligente de centroides
2. **Mini-Batch K-Means**: Versión más rápida para grandes datasets
3. **K-Medoids**: Usa puntos reales como centroides (más robusto a outliers)
4. **Fuzzy C-Means**: Asignación probabilística (soft clustering)
5. **Hierarchical Clustering**: Alternativa que no requiere especificar K

**Cuándo usar K-Means:**

✅ **Clusters esféricos y separados**: Funciona mejor  
✅ **Dataset grande**: K-Means escala bien  
✅ **Conoces aproximadamente K**: O puedes estimarlo  
✅ **Features normalizadas**: Siempre normalizar antes

❌ **NO usar cuando:**
- Clusters tienen formas irregulares o alargadas
- Diferentes densidades o tamaños de clusters
- Muchos outliers en los datos
- No tienes idea de cuántos clusters esperar

**Comparación con otros algoritmos:**

| Algoritmo | Ventaja | Desventaja |
|-----------|---------|------------|
| K-Means | Rápido, simple | Requiere K, esféricos |
| DBSCAN | Detecta formas arbitrarias | Sensible a parámetros |
| Hierarchical | No requiere K | Lento con datos grandes |
| GMM | Soft clustering, formas elípticas | Más complejo |

</div>

In [ ]:
# GRADED FUNCTION: compute_new_centroids

def compute_new_centroids(X, labels, n_clusters):
    """
    Calcula nuevos centroides como la media de puntos asignados.
    
    Parámetros
    ----------
    X : ndarray
        Puntos de datos, forma (n_samples, n_features)
    labels : ndarray
        Asignaciones de cluster, forma (n_samples,)
    n_clusters : int
        Número de clusters
    
    Retorna
    -------
    ndarray
        Nuevos centroides, forma (n_clusters, n_features)
    
    Ejemplo
    -------
    >>> X = np.array([[1, 2], [1, 4], [8, 8], [9, 10]])
    >>> labels = np.array([0, 0, 1, 1])
    >>> compute_new_centroids(X, labels, 2)
    array([[ 1.,  3.],
           [ 8.5, 9.]])
    """
    
    ### YOUR CODE STARTS HERE ###
    centroids = None
    ### YOUR CODE ENDS HERE ###
    
    return centroids

# Prueba tu implementación
X_test = np.array([[1, 2], [1, 4], [1, 0], [8, 8], [9, 10], [10, 9]])
labels_test = np.array([0, 0, 0, 1, 1, 1])

resultado = compute_new_centroids(X_test, labels_test, 2)
print(f"Nuevos centroides:\n{resultado}")
print("\nEsperado:")
print("[[1.  2. ]")
print(" [9.  9. ]]")

# Verificar con test automático
verificar_centroids = test_ejercicio_2_update_centroids()
verificar_centroids(compute_new_centroids)

<a name='ex02'></a>
### Ejercicio 2: Implementar Actualización de Centroides

Implementa la función que calcula los nuevos centroides como la media de los puntos asignados.

**Instrucciones:**
- Para cada cluster k, selecciona los puntos asignados a ese cluster
- Calcula la media de esos puntos usando `np.mean`
- Si un cluster está vacío, retorna el centroide anterior
- Retorna un array de forma (n_clusters, n_features)

In [ ]:
# Comparar K = 2, 3, 5
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_map = ['red', 'blue', 'green', 'orange', 'purple']

for idx, k_val in enumerate([2, 3, 5]):
    kmeans_compare = KMeans(n_clusters=k_val, max_iters=100, random_state=42)
    kmeans_compare.fit(X)
    
    # Graficar
    ax = axes[idx]
    for k in range(k_val):
        cluster_points = X[kmeans_compare.labels == k]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                  c=colors_map[k], alpha=0.6, edgecolors='k', s=50)
    
    # Centroides
    ax.scatter(kmeans_compare.centroids[:, 0], kmeans_compare.centroids[:, 1],
              c='black', marker='X', s=200, linewidths=2, edgecolors='yellow')
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(f'K = {k_val} (Inercia: {kmeans_compare.inertia_:.0f})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observaciones:")
print("  • K=2: Underfitting - combina clusters naturales")
print("  • K=3: Óptimo - captura la estructura real")
print("  • K=5: Overfitting - divide clusters innecesariamente")

### 5.3 - Comparar Diferentes Valores de K

Veamos cómo lucen los clusters con diferentes valores de K:

In [ ]:
# Probar diferentes valores de K
K_values = range(1, 11)
inertias = []

print("Probando diferentes valores de K...")
for k in K_values:
    kmeans_temp = KMeans(n_clusters=k, max_iters=100, random_state=42)
    kmeans_temp.fit(X)
    inertias.append(kmeans_temp.inertia_)
    print(f"  K={k}: Inercia = {kmeans_temp.inertia_:.2f}")

# Visualizar método del codo
plt.figure(figsize=(10, 6))
plt.plot(K_values, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo para determinar K óptimo')
plt.grid(True, alpha=0.3)

# Marcar el K óptimo (en este caso K=3)
optimal_k = 3
plt.axvline(x=optimal_k, color='r', linestyle='--', alpha=0.7, linewidth=2, 
            label=f'K óptimo = {optimal_k}')
plt.legend()
plt.show()

print(f"\n🎯 Observación:")
print(f"   El 'codo' aparece en K={optimal_k}")
print(f"   Después de K={optimal_k}, la inercia disminuye lentamente")
print(f"   Conclusión: K={optimal_k} es el número óptimo de clusters")

<a name='5'></a>
## 5 - Selección del Número de Clusters K

Una de las preguntas más importantes: **¿Cuántos clusters debo usar?**

### 5.1 - El Método del Codo (Elbow Method)

La idea es ejecutar K-Means con diferentes valores de K y graficar la inercia:

1. **Calcular inercia** para K = 1, 2, 3, ..., K_max
2. **Graficar** inercia vs K
3. **Buscar el "codo"**: punto donde la inercia deja de disminuir drásticamente

**Intuición:** 
- K pequeño: clusters grandes, alta inercia
- K grande: clusters pequeños, baja inercia (pero overfitting)
- El "codo" representa el mejor balance

### 5.2 - Aplicar el Método del Codo

In [ ]:
# GRADED FUNCTION: assign_to_closest_centroid

def assign_to_closest_centroid(X, centroids):
    """
    Asigna cada punto al centroide más cercano.
    
    Parámetros
    ----------
    X : ndarray
        Puntos de datos, forma (n_samples, n_features)
    centroids : ndarray
        Centroides actuales, forma (n_clusters, n_features)
    
    Retorna
    -------
    ndarray
        Índice del cluster asignado para cada punto, forma (n_samples,)
    
    Ejemplo
    -------
    >>> X = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8]])
    >>> centroids = np.array([[1, 2], [8, 8]])
    >>> assign_to_closest_centroid(X, centroids)
    array([0, 0, 1, 1])
    """
    
    ### YOUR CODE STARTS HERE ###
    labels = None
    ### YOUR CODE ENDS HERE ###
    
    return labels

# Prueba tu implementación
X_test = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8], [9, 10]])
centroids_test = np.array([[1, 2], [8, 8]])

resultado = assign_to_closest_centroid(X_test, centroids_test)
print(f"Asignaciones: {resultado}")
print("Esperado: [0, 0, 1, 1, 1]")

# Verificar con test automático
verificar_assign = test_ejercicio_1_assign_clusters()
verificar_assign(assign_to_closest_centroid)

<a name='ex01'></a>
### Ejercicio 1: Implementar Asignación de Clusters

Implementa la función que asigna cada punto al centroide más cercano.

**Instrucciones:**
- Calcula las distancias de cada punto a cada centroide
- Usa `np.linalg.norm` para calcular distancias Euclidianas
- Retorna el índice del centroide más cercano para cada punto
- Usa `np.argmin` para encontrar el mínimo

## 🎓 Resumen

- ✅ K-Means agrupa datos sin etiquetas
- ✅ Algoritmo iterativo: asignar → actualizar
- ✅ Método del codo para elegir K
- ✅ Sensible a inicialización (múltiples corridas)
- ✅ Asume clusters esféricos

### Próximo: Redes Neuronales desde Cero